# Demo Quantitative Stock Selection Model: Asset Growth

## List Comprehension

List comprehension offers a shorter syntax when you want to create a new list based on the values of an existing list. It is a way to use for-loop in a compact form. 

In [1]:
#Suppose we want to compute the square each number in mylist
mylist = [21, 30, 12, 24, 9]
mylist_square = [ 21*21, 30*30, 12*12, 24*24, 9*9]
mylist_square = [ s*s for s in mylist ]
mylist_square

[441, 900, 144, 576, 81]

## Concatenate Data Frames

Given two data frames with the same set of columns, we will stack one on topt of the other using **pd.concat()**. 

In [2]:
import pandas as pd
import numpy as np
df1 = pd.DataFrame({'key': ['A', 'B', 'C', 'D'],
                    'value': [1, 2, 3, 5]})
df2 = pd.DataFrame({'key': ['E', 'D', 'C', 'B'],
                    'value': [5, 6, 7, 8]})

display(df1)
display(df2)

,key,value
0,A,1
1,B,2
2,C,3
3,D,5


,key,value
0,E,5
1,D,6
2,C,7
3,B,8


In [3]:
#Concatenate df1 and df2 with df1 on top of df2
df3 = pd.concat([df1,df2])
df3.reset_index(drop=True, inplace=True)
df3

,key,value
0,A,1
1,B,2
2,C,3
3,D,5
4,E,5
5,D,6
6,C,7
7,B,8


## Asset Growth 

$$
\frac{\text{Total Assets at }t}{\text{Total Assets at }{t-1}}
$$

In [4]:
import pandas as pd
import numpy as np
df = pd.read_csv('classdata/AnnualTotalAsset.csv')
df['datadate']=pd.to_datetime(df['datadate'],format="%Y%m%d")

In [5]:
#We only select the columns we need
df=df[["LPERMNO","datadate","fyear","at"]]
df.head()

,LPERMNO,datadate,fyear,at
0,10421,1999-03-31,1998.0,1030.859
1,10421,2000-03-31,1999.0,1035.895
2,10421,2001-03-31,2000.0,799.893
3,10421,2001-12-31,2001.0,800.809
4,10421,2002-12-31,2002.0,801.533


Column **at** is the total asset. Column **datadate** is when the fiscal year end. Column **fyear** is the fiscal year.

Sort the dataframe by **LPERMNO** and then by **datadate**

In [6]:
df.sort_values(by=['LPERMNO', 'datadate'], inplace=True)
df.reset_index(drop=True, inplace=True)

## Step 1: Generate Signal

We introduce two new command:

   - Use **pct_change()** to calculate the percentage change between rows in total assets.
   - Use **groupby** to group rows from the same stock and apply **pct_change** to each group.

The output of **df.groupby("LPERMNO")** is a special object called "groupby" data frame. Different from a regular data frame, a groupby data frame cannot be printed.  A groupby object contains information about the groups and allows us to apply the same transformation to each group. 

In [7]:
type(df.groupby("LPERMNO"))

pandas.core.groupby.generic.DataFrameGroupBy

In [8]:
df.groupby("LPERMNO")

**get_group()** will return the rows from the same group (same stock in this case). 

In [9]:
#The rows from stock 93433
df.groupby("LPERMNO").get_group(93433)

,LPERMNO,datadate,fyear,at
6217,93433,1998-12-31,1998.0,247.073
6218,93433,1999-12-31,1999.0,235.101
6219,93433,2000-12-31,2000.0,226.909
6220,93433,2001-12-31,2001.0,275.667
6221,93433,2002-12-31,2002.0,315.451
6222,93433,2003-12-31,2003.0,412.134
6223,93433,2004-12-31,2004.0,572.927
6224,93433,2005-12-31,2005.0,512.994
6225,93433,2006-12-31,2006.0,544.063
6226,93433,2007-12-31,2007.0,475.277


Apply **.pct_change()+1** to generate the asset growth for stock  93433

In [10]:
df.groupby("LPERMNO").get_group(93433)["at"].pct_change()+1

6217         NaN
6218    0.951545
6219    0.965155
6220    1.214879
6221    1.144319
6222    1.306491
6223    1.390147
6224    0.895392
6225    1.060564
6226    0.873570
6227    0.933359
6228    0.955487
6229    0.891428
6230    0.892216
6231    1.028358
6232    0.867492
6233    1.176317
6234    0.912337
6235    1.471436
6236    0.982293
6237    1.060802
6238    0.855699
Name: at, dtype: float64

By removing **.get_group(93433)** in the code above, we can generate the asset growth for each stock. We save the asset growth in a new column called **Signal**.

In [11]:
df["Signal"]=df.groupby("LPERMNO")["at"].pct_change()+1
df.groupby("LPERMNO").get_group(93433)

,LPERMNO,datadate,fyear,at,Signal
6217,93433,1998-12-31,1998.0,247.073,NaN
6218,93433,1999-12-31,1999.0,235.101,0.951545
6219,93433,2000-12-31,2000.0,226.909,0.965155
6220,93433,2001-12-31,2001.0,275.667,1.214879
6221,93433,2002-12-31,2002.0,315.451,1.144319
6222,93433,2003-12-31,2003.0,412.134,1.306491
6223,93433,2004-12-31,2004.0,572.927,1.390147
6224,93433,2005-12-31,2005.0,512.994,0.895392
6225,93433,2006-12-31,2006.0,544.063,1.060564
6226,93433,2007-12-31,2007.0,475.277,0.873570


## Step 2: Generate Date When Each Signal Becomes Available

We now have the signal in column **Signal**. Let's generate the quarter when each signal becomes available for the first time in a new column called  **Date**. 

In this example, we assume the available date is six months after **datadate** and we represent each quarter by "yyyymm" with mm being 03, 06, 09, or 12. 

In [12]:
#These libraries are loaded to move each date six months forwards (to the end of the returned month)
from dateutil.relativedelta import relativedelta

In [13]:
#Let's generate the available date by moving a datadate 6 month forward. For example:
s=pd.to_datetime("20100831",format="%Y%m%d")
s+relativedelta(months=6)

Timestamp('2011-02-28 00:00:00')

In [14]:
#Get the year part
(s+relativedelta(months=6)).year

2011

In [15]:
#Get the quarter part
(s+relativedelta(months=6)).quarter

1

Generate **Date** as $100\times year + 3\times quarter$ using list comprehension.

In [16]:
df["year"]=[(s+relativedelta(months=6)).year for s in df["datadate"]]
df["quarter"]=[(s+relativedelta(months=6)).quarter for s in df["datadate"]]
df["Date"]=df["year"]*100+df["quarter"]*3

In [17]:
df.groupby("LPERMNO").get_group(93433)

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
6217,93433,1998-12-31,1998.0,247.073,NaN,1999,2,199906
6218,93433,1999-12-31,1999.0,235.101,0.951545,2000,2,200006
6219,93433,2000-12-31,2000.0,226.909,0.965155,2001,2,200106
6220,93433,2001-12-31,2001.0,275.667,1.214879,2002,2,200206
6221,93433,2002-12-31,2002.0,315.451,1.144319,2003,2,200306
6222,93433,2003-12-31,2003.0,412.134,1.306491,2004,2,200406
6223,93433,2004-12-31,2004.0,572.927,1.390147,2005,2,200506
6224,93433,2005-12-31,2005.0,512.994,0.895392,2006,2,200606
6225,93433,2006-12-31,2006.0,544.063,1.060564,2007,2,200706
6226,93433,2007-12-31,2007.0,475.277,0.873570,2008,2,200806


Be careful that some stock may have a growth equal to infinity or NaN. See the following examples

In [18]:
df.groupby("LPERMNO").get_group(32548)

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
1493,32548,2001-01-31,2000.0,0.000,NaN,2001,3,200109
1494,32548,2002-01-31,2001.0,0.000,NaN,2002,3,200209
1495,32548,2003-01-31,2002.0,0.000,NaN,2003,3,200309
1496,32548,2004-01-31,2003.0,11525.631,inf,2004,3,200409
1497,32548,2005-01-31,2004.0,12466.123,1.081600,2005,3,200509
1498,32548,2006-01-31,2005.0,10992.444,0.881785,2006,3,200609
1499,32548,2007-01-31,2006.0,13953.295,1.269353,2007,3,200709
1500,32548,2008-01-31,2007.0,16344.600,1.171379,2008,3,200809
1501,32548,2009-01-31,2008.0,17395.817,1.064316,2009,3,200909
1502,32548,2010-01-31,2009.0,11908.196,0.684544,2010,3,201009


To remove the abnormal data, we need to replace infinity by NaN and drop the rows that contain NaNs. 

In [19]:
df.replace(np.inf, np.nan, inplace=True)
df.dropna(how='any',axis=0, inplace=True)
df.sort_values(by=['LPERMNO', 'Date'],inplace=True)
df.reset_index(drop=True,inplace=True)
df.head()

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
0,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200009
1,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200109
2,10421,2001-12-31,2001.0,800.809,1.001145,2002,2,200206
3,10421,2002-12-31,2002.0,801.533,1.000904,2003,2,200306
4,10421,2003-12-31,2003.0,823.807,1.027789,2004,2,200406


## Step 3: Generate Signals for All Dates

Note that since the signal is constructed using the data at the annual frequency, the signal will likely be used four 
times. For example, for signals constructed using the financial statement for the fiscal year ending in December 2012, we assume the same signal value is available in June, September, and December of 2013 and in March of 2014, after which the financial statement for 2013 will be used to update the signal value.  

The code above only generates one signal each year. Next we will need to generate the signal for all quarters of a year.

The strategy is to make three copies of **df** with their **Date** being 9, 12, and 15 months forward from **datadate**. To do so, we first move **datadate** forward by 9, 12, and 15 months, respectively, and generate **Date** as $100\times year + 3\times quarter$ using list comprehension.

In [20]:
#fill the signal for the remaining three quarters of a year.
dftemp=df.copy()
for i in [9, 12, 15]:
    dftemp["Date"]=[(s+relativedelta(months=i)).year*100+(s+relativedelta(months=i)).quarter*3 for s in dftemp["datadate"]]
    df=pd.concat([df,dftemp])

df.sort_values(by=['LPERMNO', 'datadate'], inplace=True)
df.reset_index(drop=True, inplace=True)

For most stocks, this works nicely. See stock 93433 below.

In [21]:
df.groupby("LPERMNO").get_group(93433)[0:12]

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
23720,93433,1999-12-31,1999.0,235.101,0.951545,2000,2,200006
23721,93433,1999-12-31,1999.0,235.101,0.951545,2000,2,200009
23722,93433,1999-12-31,1999.0,235.101,0.951545,2000,2,200012
23723,93433,1999-12-31,1999.0,235.101,0.951545,2000,2,200103
23724,93433,2000-12-31,2000.0,226.909,0.965155,2001,2,200106
23725,93433,2000-12-31,2000.0,226.909,0.965155,2001,2,200109
23726,93433,2000-12-31,2000.0,226.909,0.965155,2001,2,200112
23727,93433,2000-12-31,2000.0,226.909,0.965155,2001,2,200203
23728,93433,2001-12-31,2001.0,275.667,1.214879,2002,2,200206
23729,93433,2001-12-31,2001.0,275.667,1.214879,2002,2,200209


Be careful that a company may change the end of its fiscal year. If that happens, there will be more than one signals on some quarters. See quarter 200206 of stock 10421 in the following example.

In [22]:
df.groupby("LPERMNO").get_group(10421)[0:7]

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
0,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200009
1,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200012
2,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200103
3,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200106
4,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200109
5,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200112
6,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200203


We use **groupby** to group the rows with the same "LPERMNO" and "Date". Then we use **.tail(1)** to only keep the last row, which is the latest signal (the one we are supposed to use).

In [23]:
df.groupby(["LPERMNO","Date"]).get_group((10421,200206))

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
7,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200206
8,10421,2001-12-31,2001.0,800.809,1.001145,2002,2,200206


In [24]:
df.groupby(["LPERMNO","Date"]).get_group((10421,200206)).tail(1)

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
8,10421,2001-12-31,2001.0,800.809,1.001145,2002,2,200206


By  removing **.get_group((10421,200206))** in the code above, we can remove duplicated Dates in all groups. 

In [25]:
df=df.groupby(["LPERMNO","Date"]).tail(1)
df.reset_index(drop=True, inplace=True)

There is only one signal in quarter 200206 now.

In [26]:
df.groupby("LPERMNO").get_group(10421)[0:7]

,LPERMNO,datadate,fyear,at,Signal,year,quarter,Date
0,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200009
1,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200012
2,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200103
3,10421,2000-03-31,1999.0,1035.895,1.004885,2000,3,200106
4,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200109
5,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200112
6,10421,2001-03-31,2000.0,799.893,0.772176,2001,3,200203


## Step 4: Generate Summary Statistics

Next, we print the summmary statistics for signals grouped by **Date** (with quarter=12 only). 

In [27]:
df[df.quarter==4].groupby("Date")["Signal"].describe(percentiles=[0.1,0.9])

,count,mean,std,min,10%,50%,90%,max
Date,,,,,,,,
199712,10.0,1.118220,0.138290,0.926448,1.005929,1.104095,1.256338,1.435548
199803,10.0,1.118220,0.138290,0.926448,1.005929,1.104095,1.256338,1.435548
199806,10.0,1.118220,0.138290,0.926448,1.005929,1.104095,1.256338,1.435548
199809,10.0,1.118220,0.138290,0.926448,1.005929,1.104095,1.256338,1.435548
199812,17.0,1.026557,0.160760,0.803513,0.832876,1.046637,1.245424,1.267813
...,...,...,...,...,...,...,...,...
201909,49.0,1.025020,0.144879,0.721118,0.827966,1.041123,1.191798,1.349212
201912,49.0,1.028295,0.138501,0.597544,0.850103,1.011194,1.183573,1.307811
202003,49.0,1.028295,0.138501,0.597544,0.850103,1.011194,1.183573,1.307811


Save the columns we want to a csv file.

In [28]:
df[["LPERMNO","datadate","Date","Signal"]].to_csv("Signal1.csv",index=False)